In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC


def build_text(df: pd.DataFrame) -> pd.Series:
    title = df["title"].fillna("")
    body = df["body"].fillna("")
    return (title + " [SEP] " + body).str.strip()


def build_models() -> list[Pipeline]:
    lr_model = Pipeline([
        ("features", FeatureUnion([
            ("word_tfidf", TfidfVectorizer(
                analyzer="word", ngram_range=(1, 2), min_df=3, max_df=0.80,
                strip_accents="unicode", lowercase=True, sublinear_tf=True,
            )),
            ("char_tfidf", TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                strip_accents="unicode", lowercase=True, sublinear_tf=True,
            )),
        ])),
        ("clf", LogisticRegression(
            C=5.0, max_iter=2500, solver="liblinear", class_weight="balanced",
        )),
    ])

    svc_base = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2), min_df=3, max_df=0.80,
            strip_accents="unicode", lowercase=True, sublinear_tf=True,
        )),
        ("clf", LinearSVC(C=1.5, class_weight='balanced', max_iter=3000, dual=False)),
    ])
    svc_model = CalibratedClassifierCV(svc_base, method="sigmoid", cv=5)  # cv=5 вместо 3
    
    return [lr_model, svc_model]


def find_best_threshold(y_true: np.ndarray, probabilities: np.ndarray) -> tuple[float, float]:
    best_threshold = 0.42
    best_f1 = -1.0
    for threshold in np.arange(0.38, 0.48, 0.005):
        pred = (probabilities >= threshold).astype(int)
        score = f1_score(y_true, pred)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def oof_ensemble_predictions(
    models: list[Pipeline], x_train, y_train, x_eval, n_splits: int = 5,
) -> tuple[np.ndarray, np.ndarray]:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof = np.zeros(len(x_train))
    eval_pred = np.zeros(len(x_eval))

    for model in models:
        model_oof = np.zeros(len(x_train))
        model_eval = np.zeros((n_splits, len(x_eval)))

        for fold_idx, (fit_idx, val_idx) in enumerate(cv.split(x_train, y_train)):
            x_fit, y_fit = x_train.iloc[fit_idx], y_train.iloc[fit_idx]
            x_val_fold = x_train.iloc[val_idx]
            fitted = clone(model)
            fitted.fit(x_fit, y_fit)
            model_oof[val_idx] = fitted.predict_proba(x_val_fold)[:, 1]
            model_eval[fold_idx] = fitted.predict_proba(x_eval)[:, 1]

        oof += model_oof / len(models)
        eval_pred += model_eval.mean(axis=0) / len(models)

    return oof, eval_pred

train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

X = build_text(train_df)
y = train_df["label"].astype(int)
X_test = build_text(test_df)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

models = build_models()
oof_val_proba, test_proba = oof_ensemble_predictions(models, X_train, y_train, X_test)
best_threshold, best_cv_f1 = find_best_threshold(y_train.to_numpy(), oof_val_proba)
print(f"OOF F1 (best threshold={best_threshold:.3f}): {best_cv_f1:.4f}")

val_proba = np.zeros(len(X_val))
for model in models:
    fitted = clone(model)
    fitted.fit(X_train, y_train)
    val_proba += fitted.predict_proba(X_val)[:, 1] / len(models)

val_pred = (val_proba >= best_threshold).astype(int)
val_f1 = f1_score(y_val, val_pred)
print(f"Validation F1: {val_f1:.4f}")

test_pred = (test_proba >= best_threshold).astype(int)

pd.DataFrame({"id": test_df["id"], "label": test_pred}).to_csv("submission.csv", index=False)

OOF F1 (best threshold=0.420): 0.8290
Validation F1: 0.8190
